# DATA CLEANING AND CREATE FEATURES

## SETUP AND IMPORT

In [1]:
import duckdb
import pandas as pd
import sqlite3
from pathlib import Path
import holidays
import os
import numpy as np
from functions import filter_train_type, historic_delay_features, create_features

os.chdir("/Users/jakoberhard/Library/CloudStorage/GoogleDrive-jakanterh@gmail.com/My Drive/uni/python/TBA_project")

con = duckdb.connect("data/train.duckdb")

con.execute("""
CREATE OR REPLACE TABLE train_delay AS
            SELECT * FROM
            read_parquet('data/train_delay_with_weather.parquet')
            """)

df_raw = con.execute("SELECT * FROM train_delay").fetchdf()

## DATA INSPECTION

In [2]:
print(df_raw.info())                      
# print(df_raw.head(5))                      
# print(df_raw["delay_in_min"].describe())  
# print(df_raw["train_type"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2175996 entries, 0 to 2175995
Data columns (total 15 columns):
 #   Column             Dtype         
---  ------             -----         
 0   ride_id            int64         
 1   train_name         object        
 2   train_type         object        
 3   station_current    object        
 4   station_dest       object        
 5   canceled           bool          
 6   arrival_planned    datetime64[us]
 7   arrival_real       datetime64[us]
 8   departure_planned  datetime64[us]
 9   departure_real     datetime64[us]
 10  delay              int32         
 11  temperature        float64       
 12  precipitation      float64       
 13  wind_speed         float64       
 14  weather_status     object        
dtypes: bool(1), datetime64[us](4), float64(3), int32(1), int64(1), object(5)
memory usage: 226.2+ MB
None


## DATA CLEANING

In [3]:
### FILTER TRAIN TYPES (ONLY ICE AND IC)
df_cleaned = filter_train_type(df_raw)

### REMOVE RIDES THAT WERE CANCELED
df_cleaned = (
    df_cleaned.groupby("ride_id")
      .filter(lambda g: not g["canceled"].any())
)

### REMOVE UNECESSARY COLUMNS
df_cleaned = df_cleaned.drop(columns=["weather_status", "canceled"])

### RENAME COLUMNS
df_cleaned = df_cleaned.rename(columns={
    "station_name": "station_current",
    "final_destination_station": "station_dest"})

# REARRANGE COLUMNS
df_cleaned = df_cleaned.loc[:, ["ride_id", "train_name", "train_type", "station_current", "station_dest",
               "departure_planned", "departure_real", "arrival_planned", "arrival_real",
                "temperature", "precipitation", "wind_speed", "delay"]]


print(df_cleaned.columns.tolist())
print(df_cleaned.info())    


['ride_id', 'train_name', 'train_type', 'station_current', 'station_dest', 'departure_planned', 'departure_real', 'arrival_planned', 'arrival_real', 'temperature', 'precipitation', 'wind_speed', 'delay']
<class 'pandas.core.frame.DataFrame'>
Index: 1842266 entries, 0 to 2175995
Data columns (total 13 columns):
 #   Column             Dtype         
---  ------             -----         
 0   ride_id            int64         
 1   train_name         object        
 2   train_type         object        
 3   station_current    object        
 4   station_dest       object        
 5   departure_planned  datetime64[us]
 6   departure_real     datetime64[us]
 7   arrival_planned    datetime64[us]
 8   arrival_real       datetime64[us]
 9   temperature        float64       
 10  precipitation      float64       
 11  wind_speed         float64       
 12  delay              int32         
dtypes: datetime64[us](4), float64(3), int32(1), int64(1), object(4)
memory usage: 189.7+ MB
None


## NA ANALYSIS

In [4]:
# number of na per column
print(df_cleaned.isnull().sum(axis = 0))

# missings in time_current_departure_planned and time_current_arrival_plannned
# are due to first or last station that do not have either arrival time (start station) or departure time (destination station)
len(df_cleaned["ride_id"].unique())

ride_id                   0
train_name                0
train_type                0
station_current           0
station_dest              0
departure_planned    200154
departure_real       200031
arrival_planned      200154
arrival_real         200036
temperature               0
precipitation             0
wind_speed                0
delay                     0
dtype: int64


200154

## CREATE FEATURES

### STATION START

In [5]:
### REASON: ALSO EXISTS IN API DATA
### AFTER THAT, WE CAN APPLY FUNCTIONS TO BOTH DATASETS

# sorting
df_cleaned.sort_values(by=["ride_id", "departure_planned"])

# grouping
grouped = df_cleaned.groupby("ride_id")

# create station_start
df_cleaned["station_start"] = grouped["station_current"].transform("first")

### HISTORICAL FEATURES

In [6]:
# extract the historical features first
# station
hist_station = historic_delay_features(
    df_cleaned,
    variable = "station_current",
    prefix = "hist_delay_station")

# train name
hist_train = historic_delay_features(
    df_cleaned,
    variable = "train_name",
    prefix = "hist_delay_train")

# create a list for merging
hist_list = [hist_station, hist_train]

### ALL OTHER FEATURES (WITH FUNCTION)

In [7]:
df_features = create_features(df = df_cleaned, 
                              api = False, 
                              historical_features = hist_list)

## GENERATE HISTORICAL DELAY LOOKUP FILE

In [8]:
# get the information for the lookup file
max_date = df_cleaned["departure_real"].max().floor("D")

# last 60 days
window_start = max_date - pd.Timedelta("60D")

# select last 60 days
df_last60 = df_cleaned[(df_cleaned["departure_real"] > window_start)]

# aggregate data
hist_delay_station = df_last60.groupby("station_current").agg(
    hist_delay_avg = ("delay", "mean"),
    hist_delay_q90 = ("delay", lambda x: x.quantile(0.9)),
    hist_delay_count = ("delay", "count")
).reset_index()

hist_delay_train = df_last60.groupby("train_name").agg(
    hist_delay_avg = ("delay", "mean"),
    hist_delay_q90 = ("delay", lambda x: x.quantile(0.9)),
    hist_delay_count = ("delay", "count")
).reset_index()

# store all in one lookup_list
hist_list_lookup = [hist_delay_station, hist_delay_train]

In [9]:
# check if all stations are in lookup file
print("Number of stations in historical dataset:", len(df_cleaned["station_current"].unique()))
print("Number of stations in lookup file:", len(hist_list_lookup[0]))

# station names that are not in lookup file
stations_diff_list = list(set(df_cleaned["station_current"].unique()) - set(hist_list_lookup[0]["station_current"]))
stations_diff_list

Number of stations in historical dataset: 299
Number of stations in lookup file: 293


['Heilbronn Hbf',
 'Berlin-Lichtenberg',
 'Berlin Friedrichstraße',
 'Bietigheim-Bissingen',
 'Kassel Hbf',
 'Tübingen Hbf']

In [10]:
# check if all trains are in lookup file
print("Number of trains in historical dataset:", len(df_cleaned["train_name"].unique()))
print("Number of trains in lookup file:", len(hist_list_lookup[1]))

# train names that are not in lookup file
trains_diff_list = list(set(df_cleaned["train_name"].unique()) - set(hist_list_lookup[1]["train_name"]))

Number of trains in historical dataset: 1340
Number of trains in lookup file: 960


In [11]:
# FUNCTION THAT PRINTS LENGTH AND TIME FRAMES FOR STATIONS
def check_len_time_frame(df, station_name):
    
    # get data for station
    temp = df[df["station_current"] == station_name]

    # numbers of observations
    n = len(temp)
    if n == 0:
        print(f"No data found for station: {station_name}")
        return

    # get time_frame of obs
    t_arr_min, t_arr_max = temp["arrival_real"].min(), temp["arrival_real"].max()
    t_dep_min, t_dep_max = temp["departure_real"].min(), temp["departure_real"].max()

    print(f"Historical dataset contains {n} ride(s) for {station_name}")
    print(f"Time frame (arrivales):   {t_arr_min} - {t_arr_max}")
    print(f"Time frame (departures): {t_dep_min} - {t_dep_max}")
    print("")

# get rides and time frames for stations that are not inlcuded in lookup file
for station in stations_diff_list:
    check_len_time_frame(df_cleaned, station)


# FUNCTION THAT STORES THE SAME INFO IN DF
def get_station_stats(df, station_names):

    temp = df[df["station_current"].isin(station_names)]

    stats_df = temp.groupby("station_current").agg(
        n_rides=("station_current", "count"),
        arrival_min=('arrival_real', "min"),
        arrival_max=('arrival_real', "max"),
        departure_min=('departure_real', "min"),
        departure_max=('departure_real', "max")
    )
    
    return stats_df

# get rides and time frames for all stations
stats_all_stations = get_station_stats(df_cleaned, df_cleaned["station_current"].unique())

Historical dataset contains 25 ride(s) for Heilbronn Hbf
Time frame (arrivales):   2024-07-19 12:15:00 - 2024-11-09 12:16:00
Time frame (departures): 2024-07-19 12:21:00 - 2024-11-09 12:21:00

Historical dataset contains 204 ride(s) for Berlin-Lichtenberg
Time frame (arrivales):   2024-07-26 22:44:00 - 2025-12-13 18:11:00
Time frame (departures): 2024-07-26 22:47:00 - 2025-10-29 17:44:00

Historical dataset contains 6 ride(s) for Berlin Friedrichstraße
Time frame (arrivales):   2024-08-08 23:23:00 - 2024-08-16 00:59:00
Time frame (departures): 2024-08-08 23:38:00 - 2024-08-16 00:59:00

Historical dataset contains 1 ride(s) for Bietigheim-Bissingen
Time frame (arrivales):   2024-10-19 22:01:00 - 2024-10-19 22:01:00
Time frame (departures): 2024-10-19 22:04:00 - 2024-10-19 22:04:00

Historical dataset contains 2 ride(s) for Kassel Hbf
Time frame (arrivales):   2025-07-20 22:19:00 - 2025-07-21 23:07:00
Time frame (departures): NaT - NaT

Historical dataset contains 207 ride(s) for Tübinge

Explaining the results:
- Tübingen Hbf: not normally connected to the long-distance transport network, Riedbahn-renovation
- Berlin Lichtenberg: not normally connected to the long-distance transport network
- Heilbronn Hbf: not normally connected to the long-distance transport network, because of Riedbahn-renovation partly ICE trains (https://www.swr.de/swraktuell/baden-wuerttemberg/heilbronn/letzter-ice-halt-in-heibronn-100.html)
- Berlin Friedrichstraße: not normally connected to the long-distance transport network
- Kassel Hbf: long-distance trains usually go via Kassel-Wilhelmshöhe
- Bietigheim-Bissingen: not normally connected to the long-distance transport network

## EXPORT DATA

In [13]:
# cleaned features
df_features.to_parquet('data/train_delay_with_features.parquet', index=False)

# lookup tables
# stations
hist_list_lookup[0].to_parquet("data/hist_delay_station_lookup.parquet", index=False)
# trains
hist_list_lookup[1].to_parquet("data/hist_delay_train_lookup.parquet", index=False)